## תא 1: התקנה

In [ ]:
!pip install -q cohere gradio numpy

cohere, numpy: כבר מותקן באתר, אין צורך להתקין.
gradio: לא רץ בדפדפן. את הממשק מריצים בתא הניסוי.


## תא 2: חיבור

In [ ]:
import json
import numpy as np
from google.colab import userdata
import cohere

co = cohere.ClientV2(userdata.get("COHERE_API_KEY"))

EMBED_MODEL = "embed-multilingual-v3.0"   # תומך עברית
CHAT_MODEL = "command-a-03-2025"
print("מחובר")

מחובר


## תא 3: המאגר

In [ ]:
with open("dataset.json", encoding="utf-8") as f:
    data = json.load(f)
questions = [item["question"] for item in data]
print(len(data), "זוגות. דוגמה:", data[0])

40 זוגות. דוגמה: {'question': 'איך מצטרפים אליכם ומניידים את המספר הקיים?', 'answer': 'ניתן להצטרף דרך אתר האינטרנט שלנו או בשיחה עם נציג מכירות. תהליך ניוד המספר מתבצע אוטומטית עם הפעלת כרטיס הסים החדש ואורך בדרך כלל עד חצי שעה.'}


## תא 4: Embeddings

In [ ]:
resp = co.embed(
    texts=questions,
    model=EMBED_MODEL,
    input_type="search_document",   # לשאלות שבמאגר
    embedding_types=["float"],
)
vectors = resp.embeddings.float_
print(len(vectors), "וקטורים, ממד", len(vectors[0]))

40 וקטורים, ממד 1024


## תא 5: שמירת הווקטורים

In [ ]:
matrix = np.array(vectors, dtype=np.float32)
matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)   # נרמול: אורך 1 לכל וקטור
np.save("faq.npy", matrix)
print("נשמר faq.npy, צורה:", matrix.shape)

נשמר faq.npy, צורה: (40, 1024)


## תא 6: שליפה

In [ ]:
def retrieve(query, k=3):
    r = co.embed(
        texts=[query],
        model=EMBED_MODEL,
        input_type="search_query",      # לשאילתה, שונה מ-search_document
        embedding_types=["float"],
    )
    q = np.array(r.embeddings.float_[0], dtype=np.float32)
    q /= np.linalg.norm(q)
    sims = matrix @ q                    # דמיון קוסינוס לכל השאלות במאגר
    top = np.argsort(-sims)[:k]          # k הגבוהים ביותר
    return [
        {"question": data[i]["question"],
         "answer": data[i]["answer"],
         "score": round(float(sims[i]), 3)}   # גבוה = דומה
        for i in top
    ]

for q in ["איבדתי את הפלאפון שלי", "כמה עולה לגלוש בחו\"ל", "מה מזג האוויר מחר?"]:
    print("\n>>", q)
    for r in retrieve(q):
        print(f"  {r['score']}  {r['question']}")


>> איבדתי את הפלאפון שלי
  0.609  אבד לי הטלפון, איך חוסמים את הקו באופן מיידי?
  0.555  אפשר להחליף את מספר הטלפון שלי למספר חדש?
  0.513  למה אני לא מצליח לשלוח או לקבל הודעות SMS?

>> כמה עולה לגלוש בחו"ל
  0.583  אני טס לחו"ל, איך רוכשים חבילת גלישה ושיחות?
  0.514  הגעתי לחו"ל ואין לי קליטה או אינטרנט, מה לעשות?
  0.497  איך יודעים אם המכשיר שרכשתי בחו"ל יעבוד ברשת שלכם?

>> מה מזג האוויר מחר?
  0.364  מה זה שירות שיחות על גבי Wi-Fi ואיך מפעילים אותו?
  0.363  האם ניתן להקפיא מנוי לתקופה זמנית?
  0.359  איפה נמצאים מרכזי השירות שלכם ומה שעות הפעילות?


## תא 7: פרומפט, Guardrails ופלט JSON

In [ ]:
SCORE_THRESHOLD = 0.45          # להחליף לפי הכיול שעשיתם
NO_ANSWER = "אני מצטער, אך המסמכים שסופקו אינם מכילים מספיק מידע כדי לענות על שאלה זו."

def build_prompt(query, results):
    context = "\n---\n".join(
        f"שאלה דומה: {r['question']}\nתשובה: {r['answer']}" for r in results
    )
    return f"""את/ה עוזר/ת שירות לקוחות. ענה על שאלת המשתמש בהתבסס *אך ורק* על ההקשר.

הקשר:
---
{context}
---

שאלה:
{query}

הוראות:
- ענה רק לפי ההקשר. אל תשתמש בידע קודם.
- אם ההקשר אינו מספיק, החזר בשדה answer בדיוק: "{NO_ANSWER}" ובשדה out_of_scope החזר true.
- היה תמציתי ואל תחזור על ההקשר מילה במילה.
- החזר JSON תקין בלבד במבנה:
{{"answer": "...", "out_of_scope": false, "confidence": "high|medium|low"}}"""

def answer(query):
    results = retrieve(query)
    if results[0]["score"] < SCORE_THRESHOLD:       # הגנה ראשונה: בלי לקרוא למודל
        return {"answer": NO_ANSWER, "out_of_scope": True,
                "confidence": "low", "sources": results}
    resp = co.chat(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": build_prompt(query, results)}],
        response_format={"type": "json_object"},
        temperature=0,
    )
    out = json.loads(resp.message.content[0].text)
    out["sources"] = results
    return out

print(json.dumps(answer("באיזו שעה אתם פתוחים?"), ensure_ascii=False, indent=2))
print(json.dumps(answer("מי ניצח במונדיאל?"), ensure_ascii=False, indent=2))

{
  "answer": "שעות הפעילות של מרכזי השירות שלנו מפורטות באתר האינטרנט תחת 'סניפים'.",
  "out_of_scope": false,
  "confidence": "high",
  "sources": [
    {
      "question": "איפה נמצאים מרכזי השירות שלכם ומה שעות הפעילות?",
      "answer": "מרכזי השירות ונקודות המכירה שלנו פרוסים בקניונים ובמרכזי קניות מרכזיים ברחבי הארץ. רשימת הסניפים המלאה, כתובות ושעות פעילות מעודכנות מופיעות באתר האינטרנט שלנו תחת 'סניפים'.",
      "score": 0.632
    },
    {
      "question": "איך ניתן ליצור קשר עם נציג שירות אנושי?",
      "answer": "ניתן לשוחח עם נציג שירות דרך הצ'אט באתר, בהודעת WhatsApp או בחיוג למוקד הטלפוני שלנו בימים א'-ה' בין השעות 08:00 ל-18:00 וביום ו' עד השעה 12:00. מוקד אובדן וגניבה פעיל 24 שעות ביממה.",
      "score": 0.494
    },
    {
      "question": "האם יש הנחות לחיילים, סטודנטים או אזרחים ותיקים?",
      "answer": "אנו מציעים מסלולים מוזלים והטבות ייעודיות לחיילים בשירות חובה, לסטודנטים ולאזרחים ותיקים. לקבלת ההטבה יש להציג תעודה מזהה רלוונטית בעת ההצטרפות או לעדכן אותה באזור

## תא 8: ניסוי חופשי (במקום Gradio)

In [ ]:
test_cases = [
    ("פגיעה טובה", "איך מצטרפים ומניידים את המספר הקיים?"),
    ("פגיעה טובה", "האם יש התחייבות או קנס יציאה בביטול מנוי?"),
    ("פגיעה טובה", "איך מפעילים כרטיס סים חדש?"),
    ("פגיעה טובה", "האם אתם תומכים בטכנולוגיית eSIM?"),
    ("פגיעה טובה", "איך חוסמים שיחות ספאם ומספרים חסויים?"),
    (
        "ניסוח חלופי",
        "גנבו לי את הנייד, מה אני עושה עכשיו כדי שלא יחייבו אותי?",
    ),
    (
        "ניסוח חלופי",
        "נגמרה לי החבילה ואני לא מצליח לגלוש, יחייבו אותי אוטומטית?",
    ),
    ("מחוץ לתחום", "מה המתכון לעוגת שוקולד?"),
]

for idx, (test_type, q) in enumerate(test_cases, start=1):
  res = answer(q)
  top_score = res["sources"][0]["score"]
  bot_answer = res["answer"]

  # בדיקה אוטומטית: שאלה מחוץ לתחום עוברת אם out_of_scope הוא True, ואחרת אם הוא False
  expected_oos = test_type == "מחוץ לתחום"
  passed = "כן" if res["out_of_scope"] == expected_oos else "לא"

  print(f"#{idx} | סוג: {test_type}")
  print(f"שאלה: {q}")
  print(f"ציון עליון: {top_score}")
  print(f"תשובת הבוט: {bot_answer}")
  print(f"עבר?: {passed}")
  print("-" * 60)

#1 | סוג: פגיעה טובה
שאלה: איך מצטרפים ומניידים את המספר הקיים?
ציון עליון: 0.767
תשובת הבוט: ניתן להצטרף דרך אתר האינטרנט שלנו או בשיחה עם נציג מכירות. תהליך ניוד המספר מתבצע אוטומטית עם הפעלת כרטיס הסים החדש ואורך בדרך כלל עד חצי שעה.
עבר?: כן
------------------------------------------------------------
#2 | סוג: פגיעה טובה
שאלה: האם יש התחייבות או קנס יציאה בביטול מנוי?
ציון עליון: 0.813
תשובת הבוט: אין התחייבות או קנס יציאה בביטול מנוי. ניתן לסיים את ההתקשרות בכל עת, והחיוב יחושב באופן יחסי עד למועד הניתוק בפועל.
עבר?: כן
------------------------------------------------------------
#3 | סוג: פגיעה טובה
שאלה: איך מפעילים כרטיס סים חדש?
ציון עליון: 0.789
תשובת הבוט: היכנסו לאזור האישי באתר או באפליקציה ולחצו על 'הפעלת סים חדש'. הזינו את מספר הטלפון ואת הספרות האחרונות המודפסות על גבי כרטיס הסים. מומלץ להפעיל מחדש את המכשיר לאחר סיום התהליך.
עבר?: כן
------------------------------------------------------------
#4 | סוג: פגיעה טובה
שאלה: האם אתם תומכים בטכנולוגיית eSIM?
ציון עליון: 0.7